# Hyperliquid trades × Fear & Greed

Association study: sentiment vs behavior and closed PnL (no leverage in CSV; use Size USD).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu

DATA = Path("../data")
ORDER = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
SENTIMENT_MAP = {n: i for i, n in enumerate(ORDER)}
LONG = {"Open Long", "Close Long", "Long > Short", "Buy"}
SHORT = {"Open Short", "Close Short", "Short > Long", "Sell"}

def position_side(d):
    if d in LONG: return "Long"
    if d in SHORT: return "Short"
    return "Other"

trades = pd.read_csv(DATA / "historical_data.csv")
sentiment = pd.read_csv(DATA / "fear_greed_index.csv")
print(trades.shape, sentiment.shape, "traders", trades["Account"].nunique())

In [ ]:
trades["date"] = pd.to_datetime(trades["Timestamp IST"], format="%d-%m-%Y %H:%M", errors="coerce").dt.date
sentiment["date_key"] = pd.to_datetime(sentiment["date"]).dt.date
sentiment["Classification"] = sentiment["classification"].str.title()
sentiment["sentiment_score"] = sentiment["Classification"].map(SENTIMENT_MAP)

m = trades.merge(
    sentiment[["date_key", "Classification", "sentiment_score"]],
    left_on="date", right_on="date_key", how="left",
)
m["closedPnL"] = m["Closed PnL"]
m["size_usd"] = m["Size USD"]
m["win"] = m["closedPnL"] > 0
m["position_side"] = m["Direction"].map(position_side)
print("Unmatched:", m["Classification"].isna().sum())

In [ ]:
by_sent = m.groupby("Classification").agg(
    trades=("closedPnL", "count"),
    avg_pnl=("closedPnL", "mean"),
    win_rate=("win", "mean"),
    avg_size=("size_usd", "mean"),
).reindex(ORDER)
by_sent

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(
    axes,
    ["avg_pnl", "win_rate", "avg_size"],
    ["Avg closed PnL", "Win rate", "Avg size USD"],
):
    by_sent[col].plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

pd.crosstab(m["Classification"], m["position_side"]).reindex(ORDER)[["Long", "Short"]].plot(
    kind="bar", stacked=True, figsize=(9, 4)
)
plt.title("Long vs short by sentiment")
plt.tight_layout()
plt.show()

In [ ]:
traders = m.groupby("Account").agg(
    total_pnl=("closedPnL", "sum"), trades=("closedPnL", "count"), win_rate=("win", "mean")
).query("trades >= 30").sort_values("total_pnl", ascending=False)
traders.head(10)

In [ ]:
f, g = m.loc[m["Classification"] == "Fear", "closedPnL"], m.loc[m["Classification"] == "Greed", "closedPnL"]
print("Mann-Whitney p:", mannwhitneyu(f, g, alternative="two-sided").pvalue)
m[["sentiment_score", "closedPnL", "size_usd"]].corr()